# Testy kalendarza

## 1. Podstawowe info
- shape, kolumny, typy danych
- zakres dat, liczba SKU

## 2. Kompletność kalendarza
- czy każde SKU ma ciągły zakres dat (bez luk)
- czy DataStart = pierwsza sprzedaż per SKU
- czy DataKoniec <= data_max_global

## 3. Dni zerowe
- ile rekordów z DokId=-1
- czy wypełnione wszystkie pola (brak NaN gdzie nie powinno być)
- czy IloscPlus=0, Wartosc=0 dla dni zerowych

## 4. Dni ze sprzedażą
- czy IloscPlus > 0 dla DokId != -1
- czy brak DokId=-1 wśród realnych transakcji

## 5. CenaPoRab
- czy zero NaN po ffill
- czy wartości sensowne (>0, brak outlierów)

## 6. Kolumny stałe per SKU
- czy NazwaTow, AsId, NazwaAsort są wypełnione dla wszystkich rekordów
- czy jeden SKU ma zawsze tę samą NazwaTow/AsId

## 7. Duplikaty
- czy są duplikaty całych rekordów
- czy są duplikaty [TowId, Data, DokId]

In [16]:
import pandas as pd
import numpy as np

kalendarz_full = pd.read_parquet("dane/interim/kalendarz_full.parquet")

In [17]:
# 1. Podstawowe info

rows, cols = kalendarz_full.shape
print(f"Rekordów: {rows:,}\nKolumn: {cols}")
print(f"\nKolumny:\n{kalendarz_full.columns.tolist()}")
print(f"\nTypy danych:\n{kalendarz_full.dtypes}")
print(f"\nUnikalnych SKU: {kalendarz_full['TowId'].nunique():,}")
print(f"Zakres dat: {kalendarz_full['Data'].min()} → {kalendarz_full['Data'].max()}")

Rekordów: 9,535,441
Kolumn: 30

Kolumny:
['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'TypPoz', 'IloscPlus', 'IloscMinus', 'CenaPoRab', 'Wartosc', 'Data', 'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty', 'Zaplacono', 'AktywnyTow', 'Dokument', 'WplywNaStan', 'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane', 'AsId', 'NazwaTow', 'EAN', 'Opis1', 'Producent', 'NazwaAsort']

Typy danych:
DokId                      int64
Kolejnosc                  Int64
NrPozycji                  Int64
TowId                      int64
TypPoz                     Int64
IloscPlus                float64
IloscMinus               float64
CenaPoRab                float64
Wartosc                  float64
Data              datetime64[ns]
KolejnyWDniu               Int64
NrDok                     object
TypDok                     int64
AktywnyDok                 int64
Razem                    float64
DoZaplaty                float64
Zaplacono                float64
AktywnyTow              

In [18]:
# 2. Kompletność kalendarza

# Czy każde SKU ma ciągły zakres dat (bez luk)
def sprawdz_luki(df):
    luki = (
        df.groupby('TowId')['Data']
        .apply(lambda x: x.sort_values().diff().dt.days.max())
    )
    return luki[luki > 1]

luki = sprawdz_luki(kalendarz_full)
print(f"SKU z lukami w kalendarzu: {len(luki)}")
if len(luki) > 0:
    print(luki.head(10))

# Czy DataStart = pierwsza sprzedaż per SKU
pierwsza_sprzedaz = (
    kalendarz_full[kalendarz_full['DokId'] != -1]
    .groupby('TowId')['Data'].min()
)
pierwsza_w_kalendarzu = kalendarz_full.groupby('TowId')['Data'].min()

roznice = (pierwsza_sprzedaz != pierwsza_w_kalendarzu).sum()
print(f"\nSKU gdzie DataStart != pierwsza sprzedaż: {roznice}")

# Czy DataKoniec <= data_max_global
data_max_global = kalendarz_full['Data'].max()
ostatni_dzien = kalendarz_full.groupby('TowId')['Data'].max()
print(f"\nSKU gdzie DataKoniec > data_max_global: {(ostatni_dzien > data_max_global).sum()}")

SKU z lukami w kalendarzu: 18044
TowId
5     371.0
15     34.0
49     49.0
53     18.0
55     19.0
56     30.0
57     15.0
58      3.0
60     27.0
61    371.0
Name: Data, dtype: float64

SKU gdzie DataStart != pierwsza sprzedaż: 0

SKU gdzie DataKoniec > data_max_global: 0


In [19]:
# 3. Dni zerowe
dni_zerowe = kalendarz_full[kalendarz_full['DokId'] == -1]
dni_realne = kalendarz_full[kalendarz_full['DokId'] != -1]

print(f"Rekordów dni zerowych: {len(dni_zerowe):,}")
print(f"Rekordów dni realnych: {len(dni_realne):,}")

# Czy wypełnione wszystkie pola
print(f"\nNaN w dniach zerowych per kolumna:")
print(dni_zerowe.isna().sum()[dni_zerowe.isna().sum() > 0])

# Czy IloscPlus=0, Wartosc=0
print(f"\nIloscPlus != 0 w dniach zerowych: {(dni_zerowe['IloscPlus'] != 0).sum()}")
print(f"Wartosc != 0 w dniach zerowych: {(dni_zerowe['Wartosc'] != 0).sum()}")

Rekordów dni zerowych: 5,428,540
Rekordów dni realnych: 4,106,901

NaN w dniach zerowych per kolumna:
Producent    3412769
dtype: int64

IloscPlus != 0 w dniach zerowych: 0
Wartosc != 0 w dniach zerowych: 0


In [20]:
# 4. Dni ze sprzedażą
print(f"IloscPlus > 0 dla DokId != -1: {(dni_realne['IloscPlus'] > 0).sum():,}")
print(f"IloscPlus = 0 dla DokId != -1: {(dni_realne['IloscPlus'] == 0).sum():,}")
print(f"\nDokId = -1 wśród realnych transakcji: {(dni_realne['DokId'] == -1).sum()}")
print(f"\nTypDok w dniach realnych:\n{(dni_realne['TypDok'].value_counts())}")

IloscPlus > 0 dla DokId != -1: 3,964,136
IloscPlus = 0 dla DokId != -1: 142,470

DokId = -1 wśród realnych transakcji: 0

TypDok w dniach realnych:
21     3373975
2       260847
50      209787
18       79612
16       51025
14       36095
100      27392
10       15192
23       14507
78       13697
126       6842
26        5984
59        5348
19        3467
88        2753
8          292
9           77
900          5
4            2
981          2
Name: TypDok, dtype: int64


In [21]:
# 5. CenaPoRab
print(f"NaN w CenaPoRab: {kalendarz_full['CenaPoRab'].isna().sum():,}")
print(f"\nStatystyki CenaPoRab:")
print(kalendarz_full['CenaPoRab'].describe().round(2))
print(f"\nCenaPoRab <= 0: {(kalendarz_full['CenaPoRab'] <= 0).sum():,}")
print(f"CenaPoRab > 1000: {(kalendarz_full['CenaPoRab'] > 1000).sum():,}")

NaN w CenaPoRab: 0

Statystyki CenaPoRab:
count    9535441.00
mean           7.40
std           10.84
min            0.00
25%            2.99
50%            4.99
75%            8.12
max         8801.00
Name: CenaPoRab, dtype: float64

CenaPoRab <= 0: 2,096
CenaPoRab > 1000: 111


In [22]:
# 6. Kolumny stałe per SKU
print("NaN w kolumnach stałych:")
for col in ['NazwaTow', 'AsId', 'NazwaAsort', 'EAN', 'Opis1']:
    nan_count = kalendarz_full[col].isna().sum()
    print(f"  {col}: {nan_count:,}")

# Czy jeden SKU ma zawsze tę samą NazwaTow i AsId
nazwy_per_sku = kalendarz_full.groupby('TowId')['NazwaTow'].nunique()
print(f"\nSKU z >1 NazwaTow: {(nazwy_per_sku > 1).sum()}")

asid_per_sku = kalendarz_full.groupby('TowId')['AsId'].nunique()
print(f"SKU z >1 AsId: {(asid_per_sku > 1).sum()}")

NaN w kolumnach stałych:
  NazwaTow: 0
  AsId: 0
  NazwaAsort: 0
  EAN: 0
  Opis1: 0

SKU z >1 NazwaTow: 0
SKU z >1 AsId: 0


In [23]:
# 8. Duplikaty
print(f"Duplikaty całych rekordów: {kalendarz_full.duplicated().sum():,}")

print(f"\nDuplikaty [TowId, Data, DokId]: {kalendarz_full.duplicated(subset=['TowId', 'Data', 'DokId']).sum():,}")

Duplikaty całych rekordów: 0

Duplikaty [TowId, Data, DokId]: 497,203


---

In [24]:
mask_dup = kalendarz_full.duplicated(subset=['TowId', 'Data', 'DokId'], keep=False)
df_dup = kalendarz_full[mask_dup]

rozne_ceny = (
    df_dup.groupby(['TowId', 'Data', 'DokId'])['CenaPoRab']
    .nunique()
)
print(f"Grup z >1 ceną: {(rozne_ceny > 1).sum():,}")
print(f"Grup z tą samą ceną: {(rozne_ceny == 1).sum():,}")

Grup z >1 ceną: 5,006
Grup z tą samą ceną: 399,371


In [25]:
mask_dup = kalendarz_full.duplicated(subset=['TowId', 'Data', 'DokId'], keep=False)
kalendarz_full[mask_dup].sort_values(['DokId', 'TowId']).head(10)[
    ['DokId', 'TowId', 'Data', 'Kolejnosc', 'NrPozycji', 'IloscPlus', 'CenaPoRab']
]

,DokId,TowId,Data,Kolejnosc,NrPozycji,IloscPlus,CenaPoRab
986128,1166199,4608,2023-01-02,4,4,1.0,2.5900
986129,1166199,4608,2023-01-02,5,5,1.0,2.5900
1782363,1166199,7713,2023-01-02,1,1,1.0,4.7900
1782364,1166199,7713,2023-01-02,2,2,1.0,4.7900
1457861,1166201,6299,2023-01-02,1,1,1.0,3.3900
1457862,1166201,6299,2023-01-02,2,2,1.0,3.3900
1477390,1166202,6308,2023-01-02,1,1,1.0,1.8900
1477391,1166202,6308,2023-01-02,2,2,1.0,1.8900
1569482,1166204,6517,2023-01-02,1,1,1.0,2.3496
1569483,1166204,6517,2023-01-02,2,2,1.0,2.3496
